# Cell type identification with scPEFT

### Single-cell Foundation Model Workshop &mdash; hands-on session

In this hands-on session you will fine-tune a **single-cell large language model
(scLLM)** to annotate cell types, using **scPEFT** &mdash; *parameter-efficient
fine-tuning* for single-cell foundation models.

By the end you will have:

1. Loaded a pretrained scGPT foundation model.
2. Preprocessed and tokenized a single-cell RNA-seq dataset.
3. Attached a lightweight **PEFT adapter** to the frozen backbone.
4. Trained a cell-type classifier and evaluated it with a confusion matrix.

> **Reference:** Fei He, Fei, Gao, Su, Zhang, Xu. *Harnessing the power of
> single-cell large language models with parameter-efficient fine-tuning using
> scPEFT.* bioRxiv 2024.01.27.577455.
> Code: https://github.com/coffee19850519/scPEFT

## Background: single-cell large language models (scLLMs)

Over the last few years a family of transformer "foundation models" has been
built for single-cell RNA-seq data &mdash; e.g. **scBERT** (2022),
**Geneformer** (2023), **scGPT** (2024), **SATURN**, and others. They share a
common recipe:

- **Pretrained on millions of cells**, learning general representations of gene
  expression.
- **Reusable across many downstream tasks** (cell-type annotation, perturbation
  prediction, batch integration, ...).
- Often improve **signal-to-noise ratio** and mitigate **batch effects**.

But they have important limitations:

- They **do not perform well zero-shot** &mdash; the pretrained model applied
  directly to a new dataset is frequently no better than simpler baselines.
- They show **no emergent "foundation-model" intelligence** on their own.
- They **need adaptation** (fine-tuning or prompting) to perform well on a
  specific dataset or condition.

The natural fix &mdash; fully fine-tuning the whole backbone &mdash; is expensive
and risks **catastrophic forgetting** of the knowledge learned during
pretraining. That is exactly the problem scPEFT solves.

## Background: parameter-efficient fine-tuning (scPEFT)

**scPEFT** freezes the large pretrained backbone (❄️) and inserts small,
**trainable adapter modules** (🔥). Only the adapters and the task head are
updated during training, which:

- ✅ **Prevents catastrophic forgetting** &mdash; the pretrained weights stay intact.
- ✅ **Minimizes computational demands** &mdash; only a tiny fraction of parameters
  are trained.

scPEFT provides several adapter designs that plug in at different points of the
model. In this notebook you select one via the `peft` configuration option
(`TOKEN` / `PREFIX` / `LORA` / `ENCODER` / `HYBRID`):

| `peft` value | Adapter | Where it plugs in | Idea |
|---|---|---|---|
| `TOKEN` | **Token Adapter** | Gene embeddings | A small down-projection &rarr; up-projection bottleneck applied to gene tokens. |
| `PREFIX` | **Prefix Adapter** | Gene embeddings | Learnable *context-aware* prefix tokens prepended to the sequence. |
| `LORA` | **LoRA** | Encoder attention ($W_Q, W_K, W_V$) | Low-rank update $B\cdot A$ with $A \sim N(0,\sigma^2)$ added to the attention projections. |
| `ENCODER` | **Encoder Adapter** | Transformer encoder blocks | A bottleneck adapter inserted after attention, before the frozen MLP/LayerNorm. |
| `HYBRID` | **Combination** | Multiple points | Uses more than one adapter type together. |

*This notebook defaults to `peft="ENCODER"`. Feel free to experiment with the
other options in the parameter cell below and compare the results.*

### What scPEFT enables and enhances

Adaptation with scPEFT **enables** applications that native scLLMs cannot handle
well &mdash; cross-species transfer, condition-specific gene-significance
analysis, and context-aware cell-group characterization &mdash; and **enhances**
existing capabilities such as domain-agnostic cell-type identification,
perturbation prediction, and batch correction. Today's session focuses on
**cell-type identification**.

---
## 1. Set up the environment

The cells below install a GPU build of PyTorch, the scientific-Python stack,
`scanpy`/`anndata`, and the `transformers` / `peft` libraries, then run a quick
sanity check. On Colab this takes a few minutes &mdash; run them top to bottom.

In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# download at https://drive.google.com/drive/folders/1cqEa56XFXNQ4TzBo1fRUB7Zd0V25f_Yc?usp=drive_link
install_path = '/content/drive/MyDrive/scPEFT_env'

# Step 2: Add the directory to Python's path in future sessions
import sys
sys.path.append(install_path)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# !pip -q uninstall -y torch torchvision torchaudio torchtext
# !pip -q cache purge

# # PyTorch GPU wheels (CUDA 12.1)
# !pip -q install --force-reinstall --target="{install_path}" --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
#   torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121

# # torchtext compatible with torch 2.3.x
# !pip -q uninstall -y torchtext
# !pip -q install --target="{install_path}" --no-cache-dir fsspec==2025.3.0 numpy==2.0 torchdata==0.11.0 requests==2.32.4 torchtext==0.18.0

# !pip -q uninstall -y torch torchvision torchaudio torchtext
# !pip -q cache purge

# # Install CUDA-enabled builds from PyPI (files.pythonhosted.org). The default
# # PyPI torch/vision/audio Linux wheels are the CUDA 12.1 builds -- equivalent to
# # the +cu121 wheels, but without depending on download.pytorch.org, which can
# # fail with SSL handshake errors on some Colab networks.
# !pip -q install --no-cache-dir --upgrade --force-reinstall --target="{install_path}"\
#   torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --no-warn-conflicts

# # torchtext compatible with torch 2.3.x (torchtext's final release)
# !pip -q install --no-cache-dir --upgrade --force-reinstall --target="{install_path}" torchtext==0.18.0 --no-warn-conflicts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 198.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 247.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 271.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 344.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 256.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 284.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 292.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 350.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 335.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 369.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 309.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 311.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# import sys
# print(f"Python version: {sys.version}")
# import torch
# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"CUDA version: {torch.version.cuda}")
# import torchtext
# # print(f"TorchText version: {torchtext.__version__}")

In [ ]:
# !pip -q install --no-cache-dir --target="{install_path}"\
#   numpy==2.0 scipy pandas==2.2.2  requests==2.32.4 rich==13.8.0 jedi==0.16 scikit-learn tqdm matplotlib seaborn \
#   h5py tables


In [ ]:
# !pip -q install --no-cache-dir --target="{install_path}"\
#   scanpy==1.12 pandas==2.2.2 numba==0.60 anndata umap-learn leidenalg igraph \
#   scib


In [ ]:
# !pip -q install --no-cache-dir --target="{install_path}"\
#   transformers==4.41.2 \
#   peft==0.9.0 \
#   accelerate==0.30.1 \
#   # sentencepiece \
#   # datasets

---
## 2. Get the scPEFT code

Clone the scPEFT repository and move into the tutorial directory. It contains
the `scgpt` package (model, tokenizer, preprocessing) and helper utilities for
attaching PEFT adapters.

In [ ]:
!git clone https://github.com/coffee19850519/scPEFT.git

fatal: destination path 'scPEFT' already exists and is not an empty directory.


In [ ]:
ls

drive/  sample_data/  scPEFT/


In [ ]:
cd scPEFT/

/content/scPEFT


In [ ]:
ls

geneformer_peft/  pipeline_scBERT/  requirements.yaml  tutorial_peft/
img/              README.md         scfoundation/
LICENSE           reproduction/     scgpt/


In [ ]:
cd tutorial_peft

/content/scPEFT/tutorial_peft


In [ ]:
ls

human_transcription_factors.txt
{install_path}/
Tutorial_BatchCorrection.ipynb
Tutorial_CellPopulationDiscovery.ipynb
Tutorial_CrossSpecies.ipynb
Tutorial_HomologeneMapping.ipynb
Tutorial_identification_Geneformer.ipynb
Tutorial_Identification.ipynb
Tutorial_identification_scBERT.ipynb
Tutorial_identification_scFoundation.ipynb
Tutorial_MarkerGeneDetection.ipynb
Tutorial_Perturbation.ipynb
Tutorial_perturbation_scFoundation.ipynb


---
## 3. Imports and setup

Import the libraries and the scPEFT/scGPT components used throughout the
notebook: the `TransformerModel`, the gene tokenizer, the `Preprocessor`, and
the PEFT utilities (`PeftConfig`, `freeze_parameters`, ...).

In [ ]:
# %%
import copy
import json
import os
from pathlib import Path
import shutil
import sys
import time
from typing import List, Tuple, Dict, Union, Optional
import warnings
import pandas as pd
import pickle
import torch
import scanpy as sc
import argparse
import seaborn as sns
import numpy as np
from scipy.sparse import issparse
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader

#Fix: Reinstall torchtext and restart the runtime to clear loaded binary libraries
try:
    import torchtext
    from torchtext._torchtext import Vocab as VocabPybind
except (ImportError, OSError):
    print("Detected torchtext compatibility issue. Reinstalling and restarting runtime...")
    !pip -q uninstall -y torchtext
    !pip -q install --no-cache-dir torchtext==0.18.0
    os.kill(os.getpid(), 9)

from torchtext.vocab import Vocab
from torchtext._torchtext import (
    Vocab as VocabPybind,
)

sys.path.insert(0, "../") # Changed from './scPEFT' to '../' to correctly point to the scPEFT root directory
import scgpt as scg
from scgpt.model import TransformerModel, AdversarialDiscriminator
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
from scgpt.loss import (
    masked_mse_loss,
    criterion_neg_log_bernoulli,
)
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor
from scgpt import SubsetsBatchSampler
from scgpt.utils import set_seed, PeftConfig, freeze_parameters, DownstreamTasks, load_pretrained

sc.set_figure_params(figsize=(6, 6))
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings('ignore')

OSError: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchtext/lib/libtorchtext.so

---
## 4. Configuration

Set the hyperparameters for this run. A few worth noting:

- **`load_model`** &mdash; path to the pretrained scGPT backbone that will be
  frozen and adapted.
- **`dataset_name`** &mdash; the dataset to fine-tune on (default `"ms"`).
- **`peft`** &mdash; which scPEFT adapter to attach (see the table above:
  `TOKEN` / `PREFIX` / `LORA` / `ENCODER` / `HYBRID`).
- **`epochs`, `lr`, `batch_size`** &mdash; standard training settings.

The **Parameter Cell** widget lets you change these interactively without editing
code &mdash; update the values and click **Apply parameters** before running the
downstream cells.

In [ ]:
#Download scGPT Human model weights at https://drive.google.com/drive/folders/1bSv7N0_6uCR5_Vc8Za1Teh6fkRTyNZTG?usp=drive_link

hyperparameter_defaults = dict(
    seed=0,
    dataset_name="NSCLC_test",
    do_train=True,
    load_model="/content/drive/MyDrive/scGPT_human",
    mask_ratio=0.0,
    epochs=6,
    n_bins=51,
    MVC=False,  # Masked value prediction for cell embedding
    ecs_thres=0.0,  # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=5e-5,
    batch_size=50,
    dropout=0.2,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=False,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene=False,
    freeze=False,  # freeze
    DSBN=False,  # Domain-spec batchnorm
    peft="ENCODER"
    # Whether using Parameter-Efficient Fine-Tuning,
    # False to disable, HYBRID/ENCODER/TOKEN/PREFIX/LORA are available for selection
)

In [ ]:
config = argparse.Namespace(**hyperparameter_defaults)
print(config)

set_seed(config.seed)

Download example dataset at https://drive.google.com/drive/folders/13Q6UcBbMZ8tLoNUMLetN8QFE0ArDBJ_G?usp=drive_link

In [ ]:
import argparse
import ipywidgets as widgets
from IPython.display import display, Markdown

display(Markdown("## Parameter Cell"))
display(Markdown(
    "Use this cell to run the notebook with your own dataset and selected settings "
    "without editing code manually. Update only the values you want, then click "
    "**Apply parameters** to rebuild the active config used by the next cells."
))

if "hyperparameter_defaults" not in globals() or hyperparameter_defaults is None:
    hyperparameter_defaults = {}

fallback_defaults = {
    "seed": 0,
    "dataset_name": "NSCLC_test",
    "do_train": True,
    "load_model": "/content/drive/MyDrive/scGPT_human",
    "mask_ratio": 0.0,
    "epochs": 6,
    "n_bins": 51,
    "MVC": False,
    "ecs_thres": 0.0,
    "dab_weight": 0.0,
    "lr": 5e-5,
    "batch_size": 50,
    "dropout": 0.2,
    "schedule_ratio": 0.9,
    "save_eval_interval": 5,
    "fast_transformer": False,
    "pre_norm": False,
    "amp": True,
    "include_zero_gene": False,
    "freeze": False,
    "DSBN": False,
    "peft": "ENCODER",
    "dataset_path": "/content/drive/MyDrive/NSCLC_test",
}

for k, v in fallback_defaults.items():
    hyperparameter_defaults.setdefault(k, v)

def _to_str(v):
    return "" if v is None else str(v)

w_seed = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("seed")),
    description="seed:",
    layout=widgets.Layout(width="320px")
)

w_dataset_name = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("dataset_name")),
    description="dataset_name:",
    layout=widgets.Layout(width="400px")
)

w_dataset_path = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("dataset_path")),
    description="dataset_path:",
    placeholder="Optional custom dataset path",
    layout=widgets.Layout(width="750px")
)

w_load_model = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("load_model")),
    description="load_model:",
    layout=widgets.Layout(width="750px")
)

w_epochs = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("epochs")),
    description="epochs:",
    layout=widgets.Layout(width="320px")
)

w_batch_size = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("batch_size")),
    description="batch_size:",
    layout=widgets.Layout(width="320px")
)

w_lr = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("lr")),
    description="lr:",
    layout=widgets.Layout(width="320px")
)

w_mask_ratio = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("mask_ratio")),
    description="mask_ratio:",
    layout=widgets.Layout(width="320px")
)

w_dropout = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("dropout")),
    description="dropout:",
    layout=widgets.Layout(width="320px")
)

w_n_bins = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("n_bins")),
    description="n_bins:",
    layout=widgets.Layout(width="320px")
)

w_peft = widgets.Text(
    value=_to_str(hyperparameter_defaults.get("peft")),
    description="peft:",
    layout=widgets.Layout(width="320px")
)

w_include_zero_gene = widgets.Dropdown(
    options=[True, False],
    value=bool(hyperparameter_defaults.get("include_zero_gene")),
    description="include_zero_gene:",
    layout=widgets.Layout(width="320px")
)

w_do_train = widgets.Dropdown(
    options=[True, False],
    value=bool(hyperparameter_defaults.get("do_train")),
    description="do_train:",
    layout=widgets.Layout(width="320px")
)

w_amp = widgets.Dropdown(
    options=[True, False],
    value=bool(hyperparameter_defaults.get("amp")),
    description="amp:",
    layout=widgets.Layout(width="320px")
)

w_freeze = widgets.Dropdown(
    options=[True, False],
    value=bool(hyperparameter_defaults.get("freeze")),
    description="freeze:",
    layout=widgets.Layout(width="320px")
)

apply_btn = widgets.Button(
    description="Apply parameters",
    button_style="success"
)

out = widgets.Output()

def _parse_value(v):
    s = str(v).strip()
    if s == "":
        return None
    if s.lower() == "true":
        return True
    if s.lower() == "false":
        return False
    try:
        if "." in s or "e" in s.lower():
            return float(s)
        return int(s)
    except:
        return s

def on_apply_clicked(b):
    global hyperparameter_defaults, config

    updates = {
        "seed": w_seed.value,
        "dataset_name": w_dataset_name.value,
        "dataset_path": w_dataset_path.value,
        "load_model": w_load_model.value,
        "epochs": w_epochs.value,
        "batch_size": w_batch_size.value,
        "lr": w_lr.value,
        "mask_ratio": w_mask_ratio.value,
        "dropout": w_dropout.value,
        "n_bins": w_n_bins.value,
        "peft": w_peft.value,
        "include_zero_gene": w_include_zero_gene.value,
        "do_train": w_do_train.value,
        "amp": w_amp.value,
        "freeze": w_freeze.value,
    }

    for key, raw_val in updates.items():
        if isinstance(raw_val, str):
            if raw_val.strip() != "":
                hyperparameter_defaults[key] = _parse_value(raw_val)
        else:
            hyperparameter_defaults[key] = raw_val

    config = argparse.Namespace(**hyperparameter_defaults)

    with out:
        out.clear_output()
        print("Applied parameters:")
        for k, v in hyperparameter_defaults.items():
            print(f"{k}: {v}")
        print("\nconfig rebuilt successfully.")

apply_btn.on_click(on_apply_clicked)

display(w_seed)
display(w_dataset_name)
display(w_dataset_path)
display(w_load_model)
display(w_epochs)
display(w_batch_size)
display(w_lr)
display(w_mask_ratio)
display(w_dropout)
display(w_n_bins)
display(w_peft)
display(w_include_zero_gene)
display(w_do_train)
display(w_amp)
display(w_freeze)
display(apply_btn)
display(out)

## Parameter Cell

Use this cell to run the notebook with your own dataset and selected settings without editing code manually. Update only the values you want, then click **Apply parameters** to rebuild the active config used by the next cells.

Text(value='0', description='seed:', layout=Layout(width='320px'))

Text(value='NSCLC_test', description='dataset_name:', layout=Layout(width='400px'))

Text(value='/content/drive/MyDrive/NSCLC_test', description='dataset_path:', layout=Layout(width='750px'), pla…

...

Dropdown(description='do_train:', layout=Layout(width='320px'), options=(True, False), value=True)

Dropdown(description='amp:', layout=Layout(width='320px'), options=(True, False), value=True)

Dropdown(description='freeze:', index=1, layout=Layout(width='320px'), options=(True, False), value=False)

Button(button_style='success', description='Apply parameters', style=ButtonStyle())

Output()

---
## 5. The cell-type identification workflow

From raw counts to an annotated cell, the model runs through four stages. scPEFT
attaches its trainable adapters (🔥) at these stages while the backbone stays
frozen (❄️):

1. **Gene tokenizer** &mdash; combines a gene *expression* layer and a gene *name*
   layer into per-gene tokens.
2. **Gene embedding** &mdash; embeds a `[CLS]` token plus each gene token.
3. **Encoder** &mdash; attention-based transformer blocks ($Q, K, V$) produce
   contextual representations.
4. **Projector** &mdash; an MLP head maps the `[CLS]` embedding to a cell type.

**Task setup for this notebook:**

- **Label:** cell type (here, under a disease condition).
- **Learning:** supervised classification.
- **Objective:** cross-entropy loss.

The next cells configure preprocessing, load the pretrained model, and run the
`Preprocessor` on the data.

In [ ]:
# settings for input and preprocessing
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
mask_ratio = config.mask_ratio
mask_value = "auto"  # for masked values, now it should always be auto

include_zero_gene = config.include_zero_gene  # if True, include zero genes among hvgs in the training
max_seq_len = 2001
n_bins = config.n_bins

# settings for parameter efficient fine tuning
assert config.peft in [False, "HYBRID", "ENCODER", "TOKEN", "PREFIX", "LORA"]
peft_config = PeftConfig(peft_type=config.peft, use_default_settings=True).to_dict()

# input/output representation
input_style = "binned"  # "normed_raw", "log1p", or "binned"
output_style = "binned"  # "normed_raw", "log1p", or "binned"

# settings for training
MLM = False  # whether to use masked language modeling, currently it is always on.
CLS = True  # celltype classification objective
ADV = False  # Adversarial training for batch correction
CCE = False  # Contrastive cell embedding objective
MVC = config.MVC  # Masked value prediction for cell embedding
ECS = config.ecs_thres > 0  # Elastic cell similarity objective
DAB = False  # Domain adaptation by reverse backpropagation, set to 2 for separate optimizer
INPUT_BATCH_LABELS = False  # TODO: have these help MLM and MVC, while not to classifier
input_emb_style = "continuous"  # "category" or "continuous" or "scaling"
cell_emb_style = "cls"  # "avg-pool" or "w-pool" or "cls"
adv_E_delay_epochs = 0  # delay adversarial training on encoder for a few epochs
adv_D_delay_epochs = 0
mvc_decoder_style = "inner product"
ecs_threshold = config.ecs_thres
dab_weight = config.dab_weight

explicit_zero_prob = MLM and include_zero_gene  # whether explicit bernoulli for zeros
do_sample_in_train = False and explicit_zero_prob  # sample the bernoulli in training

per_seq_batch_sample = False

# settings for optimizer
lr = config.lr  # TODO: test learning rate ratio between two tasks
lr_ADV = 1e-3  # learning rate for discriminator, used when ADV is True
early_stop = 15
batch_size = config.batch_size
eval_batch_size = config.batch_size
epochs = config.epochs
schedule_interval = 1

# settings for the model
fast_transformer = config.fast_transformer
fast_transformer_backend = "flash"  # "linear" or "flash"
dropout = config.dropout  # dropout probability

# logging
log_interval = 100  # iterations
save_eval_interval = config.save_eval_interval  # epochs
do_eval_scib_metrics = True

In [ ]:
# %% validate settings
assert input_style in ["normed_raw", "log1p", "binned"]
assert output_style in ["normed_raw", "log1p", "binned"]
assert input_emb_style in ["category", "continuous", "scaling"]
if input_style == "binned":
    if input_emb_style == "scaling":
        raise ValueError("input_emb_style `scaling` is not supported for binned input.")
elif input_style == "log1p" or input_style == "normed_raw":
    if input_emb_style == "category":
        raise ValueError(
            "input_emb_style `category` is not supported for log1p or normed_raw input."
        )

if input_emb_style == "category":
    mask_value = n_bins + 1
    pad_value = n_bins  # for padding gene expr values
    n_input_bins = n_bins + 2
else:
    mask_value = -1
    pad_value = -2
    n_input_bins = n_bins

if ADV and DAB:
    raise ValueError("ADV and DAB cannot be both True.")
DAB_separate_optim = True if DAB > 1 else False

In [ ]:
dataset_name = config.dataset_name
save_dir = Path(f"./save/peft_{dataset_name}-{time.strftime('%b%d-%H-%M')}/")
save_dir.mkdir(parents=True, exist_ok=True)
print(f"save to {save_dir}")
logger = scg.logger
scg.utils.add_file_handler(logger, save_dir / "run.log")

In [ ]:
from pathlib import Path
import scanpy as sc

# Use config values, which are set by the parameter cell
dataset_path = config.dataset_path
dataset_name = config.dataset_name

if str(dataset_path).strip() != "":
    data_dir = Path(dataset_path)
else:
    # The original logic for data_dir was relative to scPEFT/tutorial_peft.
    # The data files are typically in scPEFT/data.
    # So if dataset_path is empty, we derive data_dir from dataset_name.
    # We are currently in /content/scPEFT/tutorial_peft, so "../data" points to /content/scPEFT/data.
    data_root = Path("../data/celltype_annotation")
    if dataset_name == "ms":
        data_dir = data_root / "ms"
    elif dataset_name == "COVID":
        data_dir = data_root / "COVID"
    elif dataset_name == "NSCLC":
        data_dir = data_root / "NSCLC"
    elif dataset_name == "MergedMonkey":
        data_dir = data_root / "MergedMonkey"
    else:
        raise ValueError(f"Unsupported dataset_name: {dataset_name} and dataset_path is empty. "
                         "Please specify a valid dataset_path in the parameters cell or use a supported dataset_name.")

adata = sc.read(data_dir / "train.h5ad")
adata_val = sc.read(data_dir / "val.h5ad")
adata_test = sc.read(data_dir / "test.h5ad")

print(f"Shape of adata (train.h5ad) after loading: {adata.shape}")
print(f"Shape of adata_val (val.h5ad) after loading: {adata_val.shape}")
print(f"Shape of adata_test (test.h5ad) after loading: {adata_test.shape}")

adata.obs["batch_id"] = adata.obs["str_batch"] = "0"
adata_val.obs["batch_id"] = adata_val.obs["str_batch"] = "1"
adata_test.obs["batch_id"] = adata_test.obs["str_batch"] = "2"

adata.var.set_index(adata.var["gene_name"], inplace=True)
adata_val.var.set_index(adata_val.var["gene_name"], inplace=True)
adata_test.var.set_index(adata_test.var["gene_name"], inplace=True)

data_is_raw = False
filter_gene_by_counts = False
adata_test_raw = adata_test.copy()
adata = adata.concatenate((adata_val, adata_test), batch_key="str_batch")

print(f"Using data_dir: {data_dir}")
print(f"dataset_name: {dataset_name}")
print(f"dataset_path: {dataset_path}")


In [ ]:
if config.load_model is not None:
    model_dir = Path(config.load_model)
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    # Check if model directory exists and if files are present and non-empty. If not, download them.
    files_to_check = [model_config_file, model_file, vocab_file]


    vocab = GeneVocab.from_file(vocab_file)
    shutil.copy(vocab_file, save_dir / "vocab.json")
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in adata.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(adata.var["id_in_vocab"])
    logger.info(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    adata = adata[:, adata.var["id_in_vocab"] >= 0]

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    logger.info(
        f"Resume model from {model_file}, the model args will override the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]

In [ ]:
from scipy.sparse import issparse

# Convert adata.X to a dense matrix if it's sparse
if issparse(adata.X):
    adata.X = adata.X.toarray()
    print("Converted adata.X to a dense matrix.")
else:
    print("adata.X is already a dense matrix.")

In [ ]:
import importlib
import scgpt.preprocess
importlib.reload(scgpt.preprocess)
from scgpt.preprocess import Preprocessor

# set up the preprocessor, use the args to config the workflow
preprocessor = Preprocessor(
    use_key="X",  # the key in adata.layers to use as raw data
    filter_gene_by_counts=filter_gene_by_counts,  # step 1
    filter_cell_by_counts=False,  # step 2
    normalize_total=1e4,  # 3. whether to normalize the raw data and to what sum
    result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
    log1p=data_is_raw,  # 4. whether to log1p the normalized data
    result_log1p_key="X_log1p",
    subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes
    hvg_flavor="seurat_v3" if data_is_raw else "cell_ranger",
    binning=n_bins,  # 6. whether to bin the raw data and to what number of bins
    result_binned_key="X_binned",  # the key in adata.layers to store the binned data
)

print(f"\n--- Before Preprocessor splitting in nf07retBpqFB ---")
print(f"adata shape: {adata.shape}")
print(f"adata.obs['str_batch'].value_counts():\n{adata.obs['str_batch'].value_counts()}")

adata_test = adata[adata.obs["str_batch"] == "2"]
adata = adata[adata.obs["str_batch"] != "2"]

print(f"\n--- After Preprocessor splitting, before application ---")
print(f"Current adata (train+val) shape: {adata.shape}")
print(f"Current adata (train+val) adata.obs['str_batch'].value_counts():\n{adata.obs['str_batch'].value_counts()}")
print(f"Current adata_test shape: {adata_test.shape}")

preprocessor(adata, batch_key=None)
preprocessor(adata_test, batch_key=None)

print(f"\n--- After Preprocessor application ---")
print(f"adata shape after preprocessing: {adata.shape}")
print(f"adata.obs['str_batch'].value_counts() after preprocessing:\n{adata.obs['str_batch'].value_counts()}")
print(f"'X_binned' in adata.layers: {'X_binned' in adata.layers}")
print(f"adata_test shape after preprocessing: {adata_test.shape}")
print(f"'X_binned' in adata_test.layers: {'X_binned' in adata_test.layers}")

In [ ]:
input_layer_key = {  # the values of this map coorespond to the keys in preprocessing
    "normed_raw": "X_normed",
    "log1p": "X_normed",
    "binned": "X_binned",
}[input_style]

genes = adata.var["gene_name"].tolist()

# Create celltype_id and mappings from the combined adata
celltypes = adata.obs["celltype"].astype("category").values
adata.obs["celltype_id"] = adata.obs["celltype"].astype("category").cat.codes
id2type = dict(enumerate(adata.obs["celltype"].astype("category").cat.categories))
type2id = {v: k for k, v in id2type.items()}
num_types = len(id2type)

# Debugging print statement to check str_batch counts
print(f"adata.obs['str_batch'].value_counts() before final split:\n{adata.obs['str_batch'].value_counts()}")

# Split the preprocessed adata into training and validation sets
adata_train_final = adata[adata.obs["str_batch"] == "0"].copy()
adata_valid_final = adata[adata.obs["str_batch"] == "1"].copy()

print(f"adata_train_final shape: {adata_train_final.shape}")
print(f"adata_valid_final shape: {adata_valid_final.shape}")

train_celltype_labels = adata_train_final.obs["celltype_id"].values
valid_celltype_labels = adata_valid_final.obs["celltype_id"].values

# batch_ids for the training data (can be derived from adata_train_final)
batch_ids = adata_train_final.obs["batch_id"].tolist()
num_batch_types = len(set(adata.obs["batch_id"].tolist())) # Use combined adata for total batch types

train_batch_labels = adata_train_final.obs["batch_id"].values
valid_batch_labels = adata_valid_final.obs["batch_id"].values

# Assign train_data and valid_data from the newly split and preprocessed AnnData objects
train_data = (
    adata_train_final.layers[input_layer_key].A
    if issparse(adata_train_final.layers[input_layer_key])
    else adata_train_final.layers[input_layer_key]
)

valid_data = (
    adata_valid_final.layers[input_layer_key].A
    if issparse(adata_valid_final.layers[input_layer_key])
    else adata_valid_final.layers[input_layer_key]
)

In [ ]:
if config.load_model is None:
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(vocab(genes), dtype=int)

In [ ]:
tokenized_train = tokenize_and_pad_batch(
    train_data,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,  # append <cls> token at the beginning
    include_zero_gene=include_zero_gene,
)
tokenized_valid = tokenize_and_pad_batch(
    valid_data,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,
    include_zero_gene=include_zero_gene,
)
logger.info(
    f"train set number of samples: {tokenized_train['genes'].shape[0]}, "
    f"\n\t feature length: {tokenized_train['genes'].shape[1]}"
)
logger.info(
    f"valid set number of samples: {tokenized_valid['genes'].shape[0]}, "
    f"\n\t feature length: {tokenized_valid['genes'].shape[1]}"
)


In [ ]:
def prepare_data(sort_seq_batch=False) -> Tuple[Dict[str, torch.Tensor]]:
    masked_values_train = random_mask_value(
        tokenized_train["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    masked_values_valid = random_mask_value(
        tokenized_valid["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    input_gene_ids_train, input_gene_ids_valid = (
        tokenized_train["genes"],
        tokenized_valid["genes"],
    )
    input_values_train, input_values_valid = masked_values_train, masked_values_valid
    target_values_train, target_values_valid = (
        tokenized_train["values"],
        tokenized_valid["values"],
    )

    tensor_batch_labels_train = torch.from_numpy(train_batch_labels).long()
    tensor_batch_labels_valid = torch.from_numpy(valid_batch_labels).long()

    tensor_celltype_labels_train = torch.from_numpy(train_celltype_labels).long()
    tensor_celltype_labels_valid = torch.from_numpy(valid_celltype_labels).long()

    if sort_seq_batch:  # TODO: update to random pick seq source in each traning batch
        train_sort_ids = np.argsort(train_batch_labels)
        input_gene_ids_train = input_gene_ids_train[train_sort_ids]
        input_values_train = input_values_train[train_sort_ids]
        target_values_train = target_values_train[train_sort_ids]
        tensor_batch_labels_train = tensor_batch_labels_train[train_sort_ids]
        tensor_celltype_labels_train = tensor_celltype_labels_train[train_sort_ids]

        valid_sort_ids = np.argsort(valid_batch_labels)
        input_gene_ids_valid = input_gene_ids_valid[valid_sort_ids]
        input_values_valid = input_values_valid[valid_sort_ids]
        target_values_valid = target_values_valid[valid_sort_ids]
        tensor_batch_labels_valid = tensor_batch_labels_valid[valid_sort_ids]
        tensor_celltype_labels_valid = tensor_celltype_labels_valid[valid_sort_ids]

    train_data_pt = {
        "gene_ids": input_gene_ids_train,
        "values": input_values_train,
        "target_values": target_values_train,
        "batch_labels": tensor_batch_labels_train,
        "celltype_labels": tensor_celltype_labels_train,
    }
    valid_data_pt = {
        "gene_ids": input_gene_ids_valid,
        "values": input_values_valid,
        "target_values": target_values_valid,
        "batch_labels": tensor_batch_labels_valid,
        "celltype_labels": tensor_celltype_labels_valid,
    }

    return train_data_pt, valid_data_pt


# dataset
class SeqDataset(Dataset):
    def __init__(self, data: Dict[str, torch.Tensor]):
        self.data = data

    def __len__(self):
        return self.data["gene_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}


# data_loader
def prepare_dataloader(
        data_pt: Dict[str, torch.Tensor],
        batch_size: int,
        shuffle: bool = False,
        intra_domain_shuffle: bool = False,
        drop_last: bool = False,
        num_workers: int = 0,
        sampler: torch.utils.data.Sampler = None,
) -> DataLoader:
    if num_workers == 0:
        num_workers = min(len(os.sched_getaffinity(0)), batch_size // 2)

    dataset = SeqDataset(data_pt)

    if per_seq_batch_sample:
        # find the indices of samples in each seq batch
        subsets = []
        batch_labels_array = data_pt["batch_labels"].numpy()
        for batch_label in np.unique(batch_labels_array):
            batch_indices = np.where(batch_labels_array == batch_label)[0].tolist()
            subsets.append(batch_indices)
        data_loader = DataLoader(
            dataset=dataset,
            batch_sampler=SubsetsBatchSampler(
                subsets,
                batch_size,
                intra_subset_shuffle=intra_domain_shuffle,
                inter_subset_shuffle=shuffle,
                drop_last=drop_last,
            ),
            num_workers=num_workers,
            pin_memory=True,
        )
        return data_loader

    data_loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
        sampler=sampler,
    )
    return data_loader

---
## 6. Build the model and attach the PEFT adapter

Instantiate the `TransformerModel` from the pretrained weights, then use the
scPEFT utilities to **freeze the backbone** and insert the trainable adapter
selected by the `peft` option. Only the adapter and the classification head will
receive gradients &mdash; this is what keeps fine-tuning cheap and guards against
catastrophic forgetting.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    nlayers_cls=3,
    n_cls=num_types if CLS else 1,
    vocab=vocab,
    dropout=dropout,
    pad_token=pad_token,
    pad_value=pad_value,
    do_mvc=MVC,
    do_dab=DAB,
    use_batch_labels=INPUT_BATCH_LABELS,
    num_batch_labels=num_batch_types,
    domain_spec_batchnorm=config.DSBN,
    input_emb_style=input_emb_style,
    n_input_bins=n_input_bins,
    cell_emb_style=cell_emb_style,
    mvc_decoder_style=mvc_decoder_style,
    ecs_threshold=ecs_threshold,
    explicit_zero_prob=explicit_zero_prob,
    use_fast_transformer=fast_transformer,
    fast_transformer_backend=fast_transformer_backend,
    pre_norm=config.pre_norm,
    peft_config=peft_config
)
if config.load_model is not None:
    load_pretrained(model, torch.load(model_file, map_location=device), verbose=False)

pre_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())

# Freeze params
if config.peft:
    freeze_parameters(model, DownstreamTasks.Identification)

post_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())

logger.info("-" * 89)
learnable_params = {k: v for k, v in model.named_parameters() if v.requires_grad}
for k, v in learnable_params.items():
    logger.info(f"Learnable params {k} with shape {v.shape}")

logger.info("Total Pre freeze Params: %.2fM" % (pre_freeze_param_count / 1e6,))
logger.info("Total Post freeze Params: %.2fM" % (post_freeze_param_count / 1e6,))

model.to(device);

if ADV:
    discriminator = AdversarialDiscriminator(
        d_model=embsize,
        n_cls=num_batch_types,
    ).to(device)

In [ ]:
# Assign a weight to each type of cell based on the proportion of cell types to facilitate better attention to fewer cell types during training
class_num = np.unique(train_celltype_labels, return_counts=True)[1].tolist()
class_weight = torch.tensor([(1 - (x / sum(class_num))) ** 2 for x in class_num]).to(device)

criterion = masked_mse_loss
criterion_cls = nn.CrossEntropyLoss(weight=class_weight)
criterion_dab = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=lr, eps=1e-4 if config.amp else 1e-8
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, schedule_interval, gamma=config.schedule_ratio
)
if DAB_separate_optim:
    optimizer_dab = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler_dab = torch.optim.lr_scheduler.StepLR(
        optimizer_dab, schedule_interval, gamma=config.schedule_ratio
    )
if ADV:
    criterion_adv = nn.CrossEntropyLoss()  # consider using label smoothing
    optimizer_E = torch.optim.Adam(model.parameters(), lr=lr_ADV)
    scheduler_E = torch.optim.lr_scheduler.StepLR(
        optimizer_E, schedule_interval, gamma=config.schedule_ratio
    )
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr_ADV)
    scheduler_D = torch.optim.lr_scheduler.StepLR(
        optimizer_D, schedule_interval, gamma=config.schedule_ratio
    )

scaler = torch.cuda.amp.GradScaler(enabled=config.amp)

---
## 7. Train

Fine-tune for the configured number of epochs using cross-entropy loss. A
class-weighted sampler is used so that rarer cell types get adequate attention
during training. Because only the adapter is trainable, each epoch is fast and
the pretrained knowledge is preserved.

In [ ]:
def train(model: nn.Module, loader: DataLoader) -> None:
    """
    Train the model for one epoch.
    """
    model.train()
    (
        total_loss,
        total_mse,
        total_cls,
        total_cce,
        total_mvc,
        total_ecs,
        total_dab,
        total_adv_E,
        total_adv_D,
        total_zero_log_prob,
        total_mvc_zero_log_prob,
    ) = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
    total_error = 0.0
    start_time = time.time()

    num_batches = len(loader)
    for batch, batch_data in enumerate(loader):
        input_gene_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        target_values = batch_data["target_values"].to(device)
        batch_labels = batch_data["batch_labels"].to(device)
        celltype_labels = batch_data["celltype_labels"].to(device)

        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
        with torch.cuda.amp.autocast(enabled=config.amp):
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
                do_sample=do_sample_in_train,
                #generative_training=False
            )

            masked_positions = input_values.eq(mask_value)  # the postions to predict
            loss = 0.0
            metrics_to_log = {}
            if MLM:
                loss_mse = criterion(
                    output_dict["mlm_output"], target_values, masked_positions
                )
                loss = loss + loss_mse
                metrics_to_log = {"train/mse": loss_mse.item()}
            if explicit_zero_prob:
                loss_zero_log_prob = criterion_neg_log_bernoulli(
                    output_dict["mlm_zero_probs"], target_values, masked_positions
                )
                loss = loss + loss_zero_log_prob
                metrics_to_log.update({"train/nzlp": loss_zero_log_prob.item()})
            if CLS:
                loss_cls = criterion_cls(output_dict["cls_output"], celltype_labels)
                loss = loss + loss_cls
                metrics_to_log.update({"train/cls": loss_cls.item()})

                error_rate = 1 - (
                    (output_dict["cls_output"].argmax(1) == celltype_labels)
                    .sum()
                    .item()
                ) / celltype_labels.size(0)
            if CCE:
                loss_cce = 10 * output_dict["loss_cce"]
                loss = loss + loss_cce
                metrics_to_log.update({"train/cce": loss_cce.item()})
            if MVC:
                loss_mvc = criterion(
                    output_dict["mvc_output"], target_values, masked_positions
                )
                loss = loss + loss_mvc
                metrics_to_log.update({"train/mvc": loss_mvc.item()})
            if MVC and explicit_zero_prob:
                loss_mvc_zero_log_prob = criterion_neg_log_bernoulli(
                    output_dict["mvc_zero_probs"], target_values, masked_positions
                )
                loss = loss + loss_mvc_zero_log_prob
                metrics_to_log.update({"train/mvc_nzlp": loss_mvc_zero_log_prob.item()})
            if ECS:
                loss_ecs = 10 * output_dict["loss_ecs"]
                loss = loss + loss_ecs
                metrics_to_log.update({"train/ecs": loss_ecs.item()})
            if DAB:
                # try weighting and separate optimizer
                loss_dab = criterion_dab(output_dict["dab_output"], batch_labels)
                loss = loss + dab_weight * loss_dab
                metrics_to_log.update({"train/dab": loss_dab.item()})

        model.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        with warnings.catch_warnings(record=True) as w:
            warnings.filterwarnings("always")
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
                error_if_nonfinite=False if scaler.is_enabled() else True,
            )
            if len(w) > 0:
                logger.warning(
                    f"Found infinite gradient. This may be caused by the gradient "
                    f"scaler. The current scale is {scaler.get_scale()}. This warning "
                    "can be ignored if no longer occurs after autoscaling of the scaler."
                )
        scaler.step(optimizer)
        scaler.update()

        if ADV:
            # rerun the model for adversarial training
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
                do_sample=do_sample_in_train,
                #generative_training=False
            )

            # TRAINING DISCRIMINATOR
            loss_adv_D = criterion_adv(
                discriminator(output_dict["cell_emb"].detach()), batch_labels
            )
            if epoch > adv_D_delay_epochs:
                discriminator.zero_grad()
                loss_adv_D.backward()
                optimizer_D.step()

            # TRAINING ENCODER
            loss_adv_E = -criterion_adv(
                discriminator(output_dict["cell_emb"]), batch_labels
            )
            # NOTE: the loss is negative here because we want to maximize
            # the cross_entropy_loss, in other words, disguise against the discriminator
            if epoch > adv_E_delay_epochs:
                model.zero_grad()
                discriminator.zero_grad()
                loss_adv_E.backward()
                optimizer_E.step()

        total_loss += loss.item()
        total_mse += loss_mse.item() if MLM else 0.0
        total_cls += loss_cls.item() if CLS else 0.0
        total_cce += loss_cce.item() if CCE else 0.0
        total_mvc += loss_mvc.item() if MVC else 0.0
        total_ecs += loss_ecs.item() if ECS else 0.0
        total_dab += loss_dab.item() if DAB else 0.0
        total_adv_E += loss_adv_E.item() if ADV else 0.0
        total_adv_D += loss_adv_D.item() if ADV else 0.0
        total_zero_log_prob += loss_zero_log_prob.item() if explicit_zero_prob else 0.0
        total_mvc_zero_log_prob += (
            loss_mvc_zero_log_prob.item() if MVC and explicit_zero_prob else 0.0
        )
        total_error += error_rate
        if batch % log_interval == 0 and batch > 0:
            lr = scheduler.get_last_lr()[0]
            ms_per_batch = (time.time() - start_time) * 1000 / log_interval
            cur_loss = total_loss / log_interval
            cur_mse = total_mse / log_interval
            cur_cls = total_cls / log_interval if CLS else 0.0
            cur_cce = total_cce / log_interval if CCE else 0.0
            cur_mvc = total_mvc / log_interval if MVC else 0.0
            cur_ecs = total_ecs / log_interval if ECS else 0.0
            cur_dab = total_dab / log_interval if DAB else 0.0
            cur_adv_E = total_adv_E / log_interval if ADV else 0.0
            cur_adv_D = total_adv_D / log_interval if ADV else 0.0
            cur_zero_log_prob = (
                total_zero_log_prob / log_interval if explicit_zero_prob else 0.0
            )
            cur_mvc_zero_log_prob = (
                total_mvc_zero_log_prob / log_interval
                if MVC and explicit_zero_prob
                else 0.0
            )
            cur_error = total_error / log_interval
            # ppl = math.exp(cur_loss)
            logger.info(
                f"| epoch {epoch:3d} | {batch:3d}/{num_batches:3d} batches | "
                f"lr {lr:05.5f} | ms/batch {ms_per_batch:5.2f} | "
                f"loss {cur_loss:5.2f} | "
                + (f"mse {cur_mse:5.2f} | mre {cur_error:5.2f} |" if MLM else "")
                + (f"cls {cur_cls:5.2f} | " if CLS else "")
                + (f"err {cur_error:5.2f} | " if CLS else "")
                + (f"cce {cur_cce:5.2f} |" if CCE else "")
                + (f"mvc {cur_mvc:5.2f} |" if MVC else "")
                + (f"ecs {cur_ecs:5.2f} |" if ECS else "")
                + (f"dab {cur_dab:5.2f} |" if DAB else "")
                + (f"adv_E {cur_adv_E:5.2f} |" if ADV else "")
                + (f"adv_D {cur_adv_D:5.2f} |" if ADV else "")
                + (f"nzlp {cur_zero_log_prob:5.2f} |" if explicit_zero_prob else "")
                + (
                    f"mvc_nzlp {cur_mvc_zero_log_prob:5.2f} |"
                    if MVC and explicit_zero_prob
                    else ""
                )
            )
            total_loss = 0
            total_mse = 0
            total_cls = 0
            total_cce = 0
            total_mvc = 0
            total_ecs = 0
            total_dab = 0
            total_adv_E = 0
            total_adv_D = 0
            total_zero_log_prob = 0
            total_mvc_zero_log_prob = 0
            total_error = 0
            start_time = time.time()


def evaluate(model: nn.Module, loader: DataLoader, return_raw: bool = False) -> float:
    """
    Evaluate the model on the evaluation data.
    """
    model.eval()
    total_loss = 0.0
    total_error = 0.0
    total_dab = 0.0
    total_num = 0
    predictions = []
    with torch.no_grad():
        for batch_data in loader:
            input_gene_ids = batch_data["gene_ids"].to(device)
            input_values = batch_data["values"].to(device)
            target_values = batch_data["target_values"].to(device)
            batch_labels = batch_data["batch_labels"].to(device)
            celltype_labels = batch_data["celltype_labels"].to(device)

            src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
            with torch.cuda.amp.autocast(enabled=config.amp):
                output_dict = model(
                    input_gene_ids,
                    input_values,
                    src_key_padding_mask=src_key_padding_mask,
                    batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                    CLS=CLS,  # evaluation does not need CLS or CCE
                    CCE=False,
                    MVC=False,
                    ECS=False,
                    do_sample=do_sample_in_train,
                    #generative_training = False,
                )
                output_values = output_dict["cls_output"]
                loss = criterion_cls(output_values, celltype_labels)

                if DAB:
                    loss_dab = criterion_dab(output_dict["dab_output"], batch_labels)

            total_loss += loss.item() * len(input_gene_ids)
            accuracy = (output_values.argmax(1) == celltype_labels).sum().item()
            total_error += (1 - accuracy / len(input_gene_ids)) * len(input_gene_ids)
            total_dab += loss_dab.item() * len(input_gene_ids) if DAB else 0.0
            total_num += len(input_gene_ids)
            preds = output_values.argmax(1).cpu().numpy()
            predictions.append(preds)

    if return_raw:
        return np.concatenate(predictions, axis=0)

    return total_loss / total_num, total_error / total_num

In [ ]:
train_data_pt, valid_data_pt = prepare_data(sort_seq_batch=per_seq_batch_sample)

# Add a weighted sampler to the dataloader based on the number of cells in the training set
class_counts = np.unique(train_data_pt['celltype_labels'], return_counts=True)[1]
class_weights = 1.0 / class_counts[train_data_pt['celltype_labels']]
sample_weights = class_weights / np.sum(class_weights)
train_sampler = torch.utils.data.sampler.WeightedRandomSampler(sample_weights, len(train_data), replacement=True)

train_loader = prepare_dataloader(
    train_data_pt,
    batch_size=batch_size,
    shuffle=False,
    intra_domain_shuffle=True,
    drop_last=False,
    sampler=train_sampler
)
valid_loader = prepare_dataloader(
    valid_data_pt,
    batch_size=eval_batch_size,
    shuffle=False,
    intra_domain_shuffle=False,
    drop_last=False,
)

In [ ]:
best_val_loss = float("inf")
best_avg_bio = 0.0
best_model = None
patience = 0

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    if config.do_train:
        train(
            model,
            loader=train_loader,
        )
    val_loss, val_err = evaluate(
        model,
        loader=valid_loader,
    )
    elapsed = time.time() - epoch_start_time
    logger.info("-" * 89)
    logger.info(
        f"| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | "
        f"valid loss/mse {val_loss:5.4f} | err {val_err:5.4f}"
    )
    logger.info("-" * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = copy.deepcopy(model)
        best_model_epoch = epoch
        logger.info(f"Best model with score {best_val_loss:5.4f}")
        patience = 0
    else:
        patience += 1
        if patience >= early_stop:
            logger.info(f"Early stop at epoch {epoch}")
            break

    scheduler.step()
    if DAB_separate_optim:
        scheduler_dab.step()
    if ADV:
        scheduler_D.step()
        scheduler_E.step()

---
## 8. Inference and evaluation

Run the best model on the held-out test set and inspect the **confusion matrix**
of predicted vs. true cell types.

**What to expect:** in the scPEFT paper's benchmarks (e.g. the NSCLC dataset
across scBERT, Geneformer, and scGPT), the *native* (zero-shot) model performs
poorly, full *fine-tuning* helps, and **scPEFT typically matches or beats full
fine-tuning while training far fewer parameters** &mdash; often also
outperforming classical annotation tools such as `singleR` and `Seurat`. Adapted
embeddings also separate held-out cell types more cleanly (avoiding the
catastrophic forgetting seen with naive fine-tuning).

In [ ]:
# %% inference
def test(model: nn.Module, adata: DataLoader) -> float:
    all_counts = (
        adata.layers[input_layer_key].A
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
    celltypes_labels = np.array(celltypes_labels)

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids)

    tokenized_test = tokenize_and_pad_batch(
        all_counts,
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,  # append <cls> token at the beginning
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    model.eval()
    predictions = evaluate(
        model,
        loader=test_loader,
        return_raw=True,
    )

    # compute accuracy, precision, recall, f1
    from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, f1_score

    accuracy = balanced_accuracy_score(celltypes_labels, predictions)
    precision = precision_score(celltypes_labels, predictions, average="macro")
    recall = recall_score(celltypes_labels, predictions, average="macro")
    macro_f1 = f1_score(celltypes_labels, predictions, average="macro")

    logger.info(
        f"Balanced accuracy: {accuracy:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}, "
        f"Macro F1: {macro_f1:.3f}"
    )

    results = {
        "test/accuracy": accuracy,
        "test/precision": precision,
        "test/recall": recall,
        "test/macro_f1": macro_f1,
    }

    return predictions, celltypes_labels, results

In [ ]:
predictions, labels, results = test(best_model, adata_test)

save_dict = {
    "predictions": predictions,
    "labels": labels,
    "results": results,
    "id_maps": id2type
}
with open(save_dir / "results.pkl", "wb") as f:
    pickle.dump(save_dict, f)

In [ ]:
from sklearn.metrics import confusion_matrix

celltypes = list(celltypes)
for i in set([id2type[p] for p in predictions]):
    if i not in celltypes:
        celltypes.remove(i)
cm = confusion_matrix(labels, predictions)
cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
cm = pd.DataFrame(cm, index=celltypes[:cm.shape[0]], columns=celltypes[:cm.shape[1]])
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues")
plt.savefig(save_dir / "confusion_matrix.png", dpi=300)

In [ ]:
# save the model into the save_dir
torch.save(best_model.state_dict(), save_dir / "model.pt")


---
## Summary and next steps

In this session you:

- Loaded a pretrained **scGPT** single-cell foundation model,
- preprocessed and tokenized scRNA-seq data,
- attached a lightweight **scPEFT adapter** to the frozen backbone,
- trained a supervised **cell-type classifier**, and
- evaluated it with a confusion matrix.

**Try next:**

- Switch the `peft` option (`TOKEN`, `PREFIX`, `LORA`, `ENCODER`, `HYBRID`) and
  compare accuracy and training cost.
- Swap in a different dataset via the parameter cell.
- Explore other scPEFT applications: cross-species cell-type identification,
  condition-specific marker discovery, and context-aware cell-group
  characterization.

**References**

- Fei He *et al.* *Harnessing the power of single-cell large language models with
  parameter-efficient fine-tuning using scPEFT.* bioRxiv 2024.01.27.577455.
- scPEFT code: https://github.com/coffee19850519/scPEFT
- Marker set (validation example): Liu, B., Hu, X., Feng, K. *et al.* *Temporal
  single-cell tracing reveals clonal revival and expansion of precursor exhausted
  T cells during anti-PD1 therapy in lung cancer.* Nat Cancer 3, 2022:108&ndash;121.